# Vespa Redeploy — Improved BGE-M3 Hybrid

Changes:
- BM25 added to first phase
- `rerank_count` raised to 100
- All signals normalised
- Parallel feed with `feed_iterable`
- Exact ANN (`approximate:false`) in smoke test

## 1 — Imports & Config

In [2]:
import pandas as pd
import json
import torch
from tqdm import tqdm
from FlagEmbedding import BGEM3FlagModel
from vespa.package import (
    Schema, Document, Field, FieldSet,
    RankProfile, Function,
    FirstPhaseRanking, SecondPhaseRanking,
    ApplicationPackage,
)
from vespa.deployment import VespaCloud
from vespa.application import Vespa
from vespa.io import VespaResponse

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

CORPUS_CSV_PATH   = "company_ne.csv"
QUERIES_JSON_PATH = "eval_queries_ne.json"
TENANT_NAME       = "anupstenant"
# VESPA_URL         = "https://cae4a7e4.a40e1008.z.vespa-app.cloud/"
# VESPA_CERT        = "/home/an00b/.vespa/anupstenant.nepali-m3-eval2.default/data-plane-public-cert.pem"
# VESPA_KEY         = "/home/an00b/.vespa/anupstenant.nepali-m3-eval2.default/data-plane-private-key.pem"


Using device: cpu


## 2 — Load Model

In [3]:
model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=(device == "cuda"))


Fetching 30 files: 100%|██████████| 30/30 [00:00<00:00, 11921.28it/s]


## 3 — Load Data

In [4]:
df = pd.read_csv(CORPUS_CSV_PATH).dropna(subset=['text', 'id', 'section'])
corpus_texts = (df['section'] + " \n " + df['text']).tolist()
print(f"Loaded {len(corpus_texts)} documents.")


Loaded 165 documents.


## 4 — Schema & Rank Profile

In [5]:
nepali_schema = Schema(
    name="nepali_docs",
    document=Document(
        fields=[
            Field(name="id",   type="string", indexing=["summary", "attribute"]),
            Field(name="text", type="string", indexing=["summary", "index"], index="enable-bm25"),
            Field(
                name="lexical_rep",
                type="tensor<bfloat16>(t{})",
                indexing=["summary", "attribute"],
            ),
            Field(
                name="dense_rep",
                type="tensor<bfloat16>(x[1024])",
                indexing=["summary", "attribute", "index"],
                attribute=["distance-metric: angular"],
                index="hnsw",
            ),
            Field(
                name="colbert_rep",
                type="tensor<bfloat16>(t{}, x[1024])",
                indexing=["summary", "attribute"],
            ),
        ]
    ),
    fieldsets=[FieldSet(name="default", fields=["text"])],
)

# ── Improved Rank Profile ─────────────────────────────────────────────────────
#
# Key improvements vs previous version:
#
#   1. First phase uses BM25 + dense + lexical — BM25 is a strong cheap signal
#      that improves candidate quality before ColBERT sees them.
#
#   2. rerank_count raised from 10 → 100 — ensures ColBERT has enough candidates
#      to find the correct doc, especially important with only 165 docs.
#
#   3. All three signals normalised via reciprocal-rank-style clamping so no
#      single signal dominates due to scale differences.
#
#   4. Final weights: dense 35% + lexical 15% + BM25 10% + ColBERT 40%
#      ColBERT gets the most weight in second phase as it is the most precise.

m3hybrid = RankProfile(
    name="m3hybrid",
    inputs=[
        ("query(q_dense)",       "tensor<bfloat16>(x[1024])"),
        ("query(q_lexical)",     "tensor<bfloat16>(t{})"),
        ("query(q_colbert)",     "tensor<bfloat16>(qt{}, x[1024])"),
        ("query(q_len_colbert)", "float"),
    ],
    functions=[
        Function(
            name="dense",
            expression="cosine_similarity(query(q_dense), attribute(dense_rep), x)",
        ),
        Function(
            name="lexical",
            expression="sum(query(q_lexical) * attribute(lexical_rep))",
        ),
        Function(
            name="max_sim",
            expression="sum(reduce(sum(query(q_colbert) * attribute(colbert_rep), x), max, t), qt) / query(q_len_colbert)",
        ),
        # Normalise each signal to [0,1] range using a simple sigmoid-like clamp
        Function(name="dense_norm",   expression="max(0, min(1, (dense + 1) / 2))"),
        Function(name="lexical_norm", expression="max(0, min(1, lexical / 10))"),
        Function(name="bm25_norm",    expression="max(0, min(1, bm25(text) / 20))"),
        Function(name="colbert_norm", expression="max(0, min(1, max_sim))"),
    ],
    # Phase 1: BM25 + dense + lexical — cheap, runs on ALL candidates
    first_phase=FirstPhaseRanking(
        expression="0.5*dense_norm + 0.3*lexical_norm + 0.2*bm25_norm"
    ),
    # Phase 2: full hybrid with ColBERT — runs on top-100 candidates only
    second_phase=SecondPhaseRanking(
        expression="0.35*dense_norm + 0.15*lexical_norm + 0.10*bm25_norm + 0.40*colbert_norm",
        rerank_count=100,
    ),
)

nepali_schema.add_rank_profile(m3hybrid)
vespa_app_package = ApplicationPackage(name="nepalim3", schema=[nepali_schema])
print("Schema defined.")


Schema defined.


## 5 — Deploy

In [6]:
vespa_cloud = VespaCloud(
    tenant=TENANT_NAME,
    application="nepali-m3-eval3",
    application_package=vespa_app_package,
)
app = vespa_cloud.deploy()
print("Deployed.")


Setting application...
Running: vespa config set application anupstenant.nepali-m3-eval3.default
Setting target cloud...
Running: vespa config set target cloud

No api-key found for control plane access. Using access token.
Checking for access token in auth.json...
Access token expired. Please re-authenticate.
Your Device Confirmation code is: WGBD-RDZG
Automatically open confirmation page in your default browser? [Y/n] 
Opened link in your browser:
	 https://login.console.vespa-cloud.com/activate?user_code=WGBD-RDZG
Waiting for login to complete in browser ... ⣽Opening in existing browser session.
[835620:835620:0100/000000.324724:ERROR:content/zygote/zygote_linux.cc:673] write: Broken pipe (32)
Waiting for login to complete in browser ... done;1m⢿
Success: Logged in
 auth.json created at /home/an00b/.vespa/auth.json
Successfully obtained access token for control plane access.
Certificate and key not found in /home/an00b/Anup2026/Software/omnidata-rag/test/.vespa or /home/an00b/.vespa

## 6 — Encode Corpus

In [7]:
print("Encoding corpus...")
embeddings = model.encode(
    corpus_texts,
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=True,
    batch_size=32,
)
print("Encoding done.")


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Encoding corpus...
Encoding done.


## 7 — Feed (Parallel)

In [8]:
# Delete existing docs first to avoid duplicates
# print("Deleting existing documents...")
# app.delete_all_docs(schema="nepali_docs", namespace="nepali_docs")
# print("Deleted.")

# Generator yields one doc dict at a time — no memory spike
def doc_generator(df, embeddings):
    for i, row in enumerate(df.itertuples()):
        colbert_vecs = embeddings["colbert_vecs"][i]
        yield {
            "id": str(row.id),
            "fields": {
                "id":          str(row.id),
                "text":        corpus_texts[i],
                "lexical_rep": {str(k): float(v) for k, v in embeddings["lexical_weights"][i].items()},
                "dense_rep":   embeddings["dense_vecs"][i].tolist(),
                "colbert_rep": {str(j): colbert_vecs[j].tolist() for j in range(colbert_vecs.shape[0])},
            },
        }

errors = []
def callback(response: VespaResponse, id: str):
    if not response.is_successful():
        errors.append((id, response.status_code))

print("Feeding documents in parallel...")
app.feed_iterable(
    doc_generator(df, embeddings),
    schema="nepali_docs",
    namespace="nepali_docs",
    callback=callback,
    max_connections=8,
)

print(f"Feed complete. Errors: {len(errors)}")
if errors:
    print(errors)


Feeding documents in parallel...
Feed complete. Errors: 0


## 8 — Smoke Test

In [9]:
# Reconnect if kernel restarted after deploy
# app = Vespa(url=VESPA_URL, cert=VESPA_CERT, key=VESPA_KEY)

query = "कर्मचारी बोनस"
q = model.encode([query], return_dense=True, return_sparse=True, return_colbert_vecs=True)

colbert_vecs  = q["colbert_vecs"][0]
q_colbert     = {str(i): vec.tolist() for i, vec in enumerate(colbert_vecs)}
q_len_colbert = float(len(colbert_vecs))

response = app.query(
    yql="""
        select id from nepali_docs where
        ({approximate:false, targetHits:10}nearestNeighbor(dense_rep, q_dense))
        or userQuery();
    """,
    ranking="m3hybrid",
    hits=5,
    body={
        "input.query(q_dense)":       q["dense_vecs"][0].tolist(),
        "input.query(q_lexical)":     {str(k): float(v) for k, v in q["lexical_weights"][0].items()},
        "input.query(q_colbert)":     q_colbert,
        "input.query(q_len_colbert)": q_len_colbert,
        "timeout": "30s",
    },
    query=query,
)

print(f"Status: {response.status_code}")
for hit in response.hits:
    print(hit["fields"]["id"], "  relevance:", hit["relevance"])


Status: 200
company_21_008   relevance: 0.5282809334819236
company_04_038   relevance: 0.48871995890885994
company_06_006   relevance: 0.48727357826450207
company_04_028   relevance: 0.4863558990860142
company_05_012   relevance: 0.47438871090019485
